# Deep Learning Project - 2025

### Università di Trento

**Authors:**  
- Antonio N. Bruno (ID: 258035)  
- Edoardo Di Tommaso (ID: 258433)



## Introduction

In recent years, large-scale Vision-Language Models (VLMs) such as CLIP [1] have demonstrated remarkable zero-shot classification performance by aligning images and text in a shared embedding space. This architecture enables flexible, prompt-driven recognition without any task-specific fine-tuning. However, when applied to fine-grained classification tasks, such as distinguishing between species of flowers, birds, or aircraft, the model’s performance tends to degrade, especially in low-data regimes.

This limitation has motivated growing interest in **few-shot adaptation**, where the goal is to adapt a pre-trained VLM to a new classification task using only a small number of labeled examples per class. In this setup, the model is trained on a few samples (shots) from a set of base classes, and then evaluated on both the base and a disjoint set of novel classes. The central challenge is to improve accuracy on base classes without sacrificing generalization to novel ones, a setting often referred to as **base-to-novel generalization**.

Our initial objective was to develop and test a few-shot adaptation method for CLIP on the Oxford Flowers dataset [2]. We experimented with several parameter-efficient fine-tuning (PEFT) approaches, including CoOp [3], CoCoOp [4], and KgCoOp[5]. However, during our investigation we uncovered a surprising finding: a substantial performance bottleneck was caused not by the model’s capacity, but by the **mismatch between dataset class names and CLIP’s training vocabulary**. By carefully aligning class labels with more natural or commonly used names, we significantly improved zero-shot accuracy, surpassing the gains obtained through fine-tuning. This suggests that label engineering can be as impactful as sophisticated adaptation methods.

In this notebook, we present the full workflow of our project: from baseline zero-shot evaluation with original class names, to systematic label alignment experiments, and finally to comparisons with state-of-the-art PEFT methods. Alongside code cells, we provide results, tables, figures, and discussion to make the report fully self-contained and reproducible.

### Imports and utilities for data handling

The code cells below import necessary libraries and define utility functions for loading datasets, processing images, and evaluating model performance.

In [ ]:
%pip install openai_clip
import clip 
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.
    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include
    Returns:
        subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

## Baseline: CLIP Zero-Shot performance

As a starting point, we evaluated the zero-shot performance of CLIP ViT-B/16 [1] on the Oxford Flowers dataset [2] using the original class names provided in the dataset.
The evaluation was conducted by constructing natural language prompts in the standard CLIP format for Oxford Flowers *“a photo of {class_name}, a type of flower”*, and computing top-1 accuracy separately on base and novel categories.

In [ ]:
# load CLIP. We will use Vit-B/16 as the visual backbone
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device) # preprocess contains CLIP's pre-defined augmentations

# define and inspect base and novel classes
# the class names listed below are the official ones
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
                "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
                "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
                "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
                "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
                "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
                "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup",
                "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
                "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan",
                "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris",
                "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple",
                "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
                "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia", "cyclamen",
                "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss", "foxglove",
                "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
                "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
# zero-shot predictions 

@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # here we apply the standard CLIP template used for Oxford Flowers to all categories
    # and immediately tokenize each sentence
    text_inputs = clip.tokenize(
        [f"a photo of a {CLASS_NAMES[c]}, a type of flower." for c in categories]
    ).to(device)

    # we can encode the text features once as they are shared for all images
    # therefore we do it outside the evaluation loop
    text_features = model.encode_text(text_inputs)
    # and here we normalize them (standard pratice with CLIP)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    # Track per-class predictions and targets
    per_class_correct = torch.zeros(len(categories))
    per_class_total = torch.zeros(len(categories))
    
    for image, target in tqdm(dataloader, desc=label):
        # base categories range from 0 to 50, whil novel ones from 51 to 101
        # therefore we must map categories to the [0, 50], otherwise we will have wrong predictions
        # Map targets in contiguous set starting from zero
        # Labels needs to be .long() in pytorch
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        # forward image through CLIP image encoder
        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # here cosine similarity between image and text features and keep the argmax for every row (every image)
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        # now we check which are correct, and sum them (False == 0, True == 1)
        correct_batch = (predicted_class == target)
        correct_predictions += correct_batch.sum().item()
        
        # Update per-class statistics
        for i in range(len(target)):
            class_idx = target[i].item()
            per_class_total[class_idx] += 1
            if correct_batch[i]:
                per_class_correct[class_idx] += 1

    # and now we compute the accuracy
    accuracy = correct_predictions / len(dataset)
    
    # Compute per-class accuracies
    per_class_accuracies = []
    for i in range(len(categories)):
        if per_class_total[i] > 0:
            class_acc = per_class_correct[i] / per_class_total[i]
            per_class_accuracies.append(class_acc.item())
        else:
            per_class_accuracies.append(0.0)
    
    return accuracy, per_class_accuracies

base_accuracy, base_per_class_acc = eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy, novel_per_class_acc = eval(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")

print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")


Despite being reasonably strong overall, performance varies significantly across classes.
By inspecting per-class accuracies, we stumbled upon a key insight: several classes show notably low accuracy.

## References

[1] Radford et Al. Learning transferable visual models from natural language supervision. In ICML, 2021. https://arxiv.org/abs/2103.00020

[2] Nilsback, M.-E. and Zisserman, A. Automated flower classification over a large number of classes. In Indian Conference on Computer Vision, Graphics and Image Processing, 2008. https://ieeexplore.ieee.org/document/4756141

[3] Zhou et Al. Learning to prompt for vision-language models. In ICCV, 2021. https://arxiv.org/abs/2109.01134

[4] Zhou et Al. Conditional prompt learning for vision-language models. In NeurIPS, 2022. https://arxiv.org/abs/2203.05557

[5] Yao et Al. Visual-Language Prompt Tuning with Knowledge-guided Context Optimization In CVPR, 2023. https://arxiv.org/abs/2303.13283